# 参考文献
https://langfuse.com/integrations/frameworks/langchain

In [1]:
from langfuse import observe, get_client, propagate_attributes, Langfuse
from langfuse.langchain import CallbackHandler
from config import llm_think, llm_instruct
import os

os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-34fb8f1a-302b-4994-95b6-651723563d4c"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-9012c5bb-b369-4fc7-adb1-668743d77cb9"
os.environ["LANGFUSE_BASE_URL"] = "http://192.168.10.5:13000"

# 初始化 Langfuse handler
langfuse_handler = CallbackHandler()
langfuse_client = get_client()

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Session / Trace / Span / Generation 概念与示例

## 产出结构

```
SESSION: chat-session-123
│
├── TRACE 1: handle_chat_turn ("What is 42 + 58?")
│   ├── SPAN: retrieve_docs
│   ├── SPAN: analyze_query
│   │   ├── SPAN: extract-and-score
│   │   │   ├── GENERATION: llm-extract-intent
│   │   │   └── SPAN: compute-confidence
│   │   └── SPAN: summarize
│   └── SPAN: format_response
│
├── TRACE 2: handle_chat_turn ("Can you explain that?")
│   ├── SPAN: retrieve_docs
│   ├── SPAN: analyze_query
│   │   ├── SPAN: extract-and-score
│   │   │   ├── GENERATION: llm-extract-intent
│   │   │   └── SPAN: compute-confidence
│   │   └── SPAN: summarize
│   └── SPAN: format_response
│
└── TRACE 3: handle_chat_turn ("Thanks!")
    ├── SPAN: retrieve_docs
    ├── SPAN: analyze_query
    │   ├── SPAN: extract-and-score
    │   │   ├── GENERATION: llm-extract-intent
    │   │   └── SPAN: compute-confidence
    │   └── SPAN: summarize
    └── SPAN: format_response
```

## Session / Trace / Span / Generation 区别

| | **Session** | **Trace** | **Span** | **Generation** |
|---|---|---|---|---|
| **是什么** | 一次完整的用户交互（如一个聊天线程） | 用户一次请求到回复的全过程 | Trace 内的一个逻辑步骤 | 专门记录 LLM 调用的 Span |
| **创建方式** | 通过 `propagate_attributes(session_id=...)` 将当前 trace 和所有子trace 关联 | `@observe()` 最外层调用 | `@observe()` 内层调用 或 `start_as_current_observation(as_type="span")` | `start_as_current_observation(as_type="generation")` |
| **数量关系** | 1 : N trace | 1 : N span | 可嵌套子 span 和 generation | 通常为叶子节点 |
| **包含什么** | 所有 trace 的完整回放 | 所有子 observation 的总耗时、input/output | 名称、input/output | 模型名、token 用量、模型参数 |
| **典型场景** | 聊天线程、多轮对话、工作流 | 用户发一条消息并收到回复 | "检索文档"、"计算"、"格式化" | GPT-4o 调用、embedding 调用 |

### 一句话总结

> **Session 是完整的故事，Trace 是故事中的一个回合，Span 是回合中的一个环节，Generation 是环节中 AI 说话的部分。** 纯逻辑代码不产生任何观测，除非你主动把它包进一个 span。

> 值得注意的是span之间可以互相嵌套，这让查看trace变得有条理和结构清晰

> 虽然下面是一个session=一个trace，但是涉及到多agent协作，background agent，子图嵌套之类的情况，更希望一个agent/子图一个trace，这个时候就可以写多个最外层的observe而不是都嵌套到一个里面去。

In [2]:
# langfuse要求trace_id必须是16位随机数字/字母的组合，所以这里用这种方式生成确定的trace_id方便查询
# 如果不指定，就会随机生成一个
predefined_trace_id = Langfuse.create_trace_id(seed="test_trace123")
print(f"trace_id={predefined_trace_id}")

@observe()
# 只有最外层的observe才是一个trace
def handle_chat_turn(user_input: str):
    with propagate_attributes(
        session_id="chat-session-123",
        user_id="user-456",
        tags=["production"],
        metadata={"channel": "web"},
    ):
        docs = retrieve_docs(user_input)
        analysis = analyze_query(docs, user_input)
        response = format_response(analysis)
        return response

@observe()
# 嵌套的子observe（内层调用）只是一个span
def retrieve_docs(query: str):
    return ["doc1: ...", "doc2: ..."]

@observe()
# 内层observe
def analyze_query(docs: list, user_input: str):
    with langfuse_client.start_as_current_observation(as_type="span", name="extract-and-score") as extract_span:
    # with langfuse_client.start_as_current_observation等价于@observe，内层observe就是span
        with langfuse_client.start_as_current_observation(as_type="generation", name="llm-extract-intent") as gen:
            # generation本质就是一种特殊的span
            gen.update(
                input={"prompt": f"Extract intent from: {user_input}"},
                output={"text": "intent: math_question"},
                model="gpt-4o",
                usage={"input_tokens": 20, "output_tokens": 8},
            )

        with langfuse_client.start_as_current_observation(as_type="span", name="compute-confidence") as score_span:
            confidence = 0.92
            score_span.update(
                input={"docs": docs},
                output={"confidence": confidence},
            )
        # span内部可以嵌套span
        # llm-extract-intent和compute-confidence是extract-and-score这个span的子span
        # 下面的summarize span则是和extract-and-score平级的另一个span

    with langfuse_client.start_as_current_observation(as_type="span", name="summarize") as summary_span:
        summary = f"Intent: math_question, confidence: {confidence}"
        summary_span.update(
            input={"docs": docs},
            output={"summary": summary},
        )

    return {"intent": "math_question", "confidence": confidence, "summary": summary}

@observe()
# 内层observe
def format_response(analysis: dict):
    return f"Based on analysis ({analysis['summary']}), here is your answer."

handle_chat_turn("What is 42 + 58?", langfuse_trace_id=predefined_trace_id)
handle_chat_turn("Can you explain that?")
handle_chat_turn("Thanks!")
# 这三个不同的trace都同属于一个session_id和一个user_id

trace_id=8caf0298bc440e092a5038857c80a089


'Based on analysis (Intent: math_question, confidence: 0.92), here is your answer.'

## 1. 如何在langchain/langgraph中使用langfuse
不同于直接在代码里调用start_as_current_observation，在langchain/langgraph中我们可以直接传入langfuse_client和session_id，框架会自动帮我们把每个agent call都包成一个span，并且把同一个session_id的span关联到同一个trace里去。

In [ ]:
from langchain.agents import create_agent


def add_numbers(a: int, b: int) -> int:
    """将两个数字相加并返回结果。"""
    return a + b

agent = create_agent(
    model=llm_instruct,
    tools=[add_numbers],
    system_prompt="你是一个乐于助人的数学导师，可以使用提供的工具进行计算。",
)

# 运行 agent
agent.invoke(
    {"messages": [{"role": "user", "content": "1242 + 8958 等于多少？"}]},
    config={"callbacks": [langfuse_handler]}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
@observe() # 自动将函数记录为一个 trace 到 Langfuse
def process_user_query(user_input: str):
    # langfuse = get_client()

    # 将 trace 属性传播到所有子观测
    with propagate_attributes(
        trace_name="user-query-processing",
        session_id="session-1234",
        user_id="user-5678",
    ):

        # 你的 Langchain 代码 - 将嵌套在 @observe trace 下
        prompt = ChatPromptTemplate.from_template("回答：{input}")
        chain = prompt | llm_instruct

        result = chain.invoke({"input": user_input}, config={"callbacks": [langfuse_handler]})
        result = chain.invoke({"input": user_input}, config={"callbacks": [langfuse_handler]})
        print(result)
    return result.content

# 使用示例
answer = process_user_query("法国的首都是哪里？")
print(answer)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 通过 Langfuse span 创建一个 trace，并在其中使用 Langchain
# 这里等价于@observe，只有最外层的是 trace，里面都是 span
with langfuse_client.start_as_current_observation(as_type="span", name="multi-step-process") as root_span:
    # 更新 trace 属性
    with propagate_attributes(
        session_id="session-1234",
        user_id="user-5678",
    ):

        # 步骤 1：初始处理（自定义逻辑）
        with langfuse_client.start_as_current_observation(as_type="span", name="input-preprocessing") as prep_span:
            processed_input = "简化版：解释量子计算"
            prep_span.update(output={"processed_query": processed_input})

        # 步骤 2：LangChain 处理
        prompt = ChatPromptTemplate.from_template("回答这个问题：{input}")
        chain = prompt | llm_instruct

        result = chain.invoke(
            {"input": processed_input},
            config={"callbacks": [langfuse_handler]}
        )
        result = chain.invoke(
            {"input": processed_input},
            config={"callbacks": [langfuse_handler]}
        )

        # 步骤 3：后处理（自定义逻辑）
        # 简单来说就是在 trace 最后加了一个 output-postprocessing 的 span
        with langfuse_client.start_as_current_observation(as_type="span", name="output-postprocessing") as post_span:
            final_result = f"回复：{result.content}"
            post_span.update(output={"final_response": final_result})